# Day 45: Make an AI Grade Another AI

You built an AI. How do you *know* it's any good? You can't eyeball a thousand answers by hand.

Today we build an **eval**: a second AI — the *judge* — scores our app's answers against a set of known-correct ones. This is how real teams measure LLMs. And we'll watch the judge catch a wrong answer live.

## Install

In [ ]:
%pip install langchain-google-genai langchain-ollama --quiet

## Setup — Two AIs

The **student** is yesterday's tiny local model (`gemma3:1b`) — the thing being tested. The **judge** is a stronger cloud model (`gemini-2.5-flash`) — the thing doing the grading. Using a stronger model to grade a weaker one is the standard setup.

In [8]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"].strip()

student = ChatOllama(model="gemma3:1b", temperature=0)        # the app under test (local, Day 44)
judge = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)  # the grader
print("✅ Student (local) + Judge (cloud) ready")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Student (local) + Judge (cloud) ready


## The Golden Dataset

Evals need a *ground truth*: questions paired with known-correct **reference** answers. This is the ruler we measure against.

In [9]:
golden = [
    {"q": "What is the capital of Australia?", "ref": "Canberra"},
    {"q": "How many letter 'e's are in the word 'excellence'?", "ref": "4 (four)"},
    {"q": "In what year did World War II end?", "ref": "1945"},
    {"q": "What is the chemical symbol for gold?", "ref": "Au"},
]
print(f"{len(golden)} graded questions, each with a correct answer")

4 graded questions, each with a correct answer


## The Judge

A second LLM with one job: compare the student's answer to the reference and return **PASS** or **FAIL**. Nothing else — a clean, parseable verdict our code can count.

In [10]:
judge_prompt = ChatPromptTemplate.from_template(
    "You are a strict grader. Compare the student's answer to the reference answer.\n"
    "Question: {q}\nReference: {ref}\nStudent answer: {ans}\n\n"
    "Reply ONLY one word: PASS if the student answer is correct, FAIL if not."
)
judge_chain = judge_prompt | judge | StrOutputParser()

def grade(q, ref, ans):
    verdict = judge_chain.invoke({"q": q, "ref": ref, "ans": ans}).strip().upper()
    return "PASS" if ("PASS" in verdict and "FAIL" not in verdict) else "FAIL"

print("✅ Judge ready")

✅ Judge ready


## Run the Eval

For every question: the student answers, the judge grades. We tally a score — automatically, no human reading each one.

In [11]:
answer_chain = ChatPromptTemplate.from_template("Answer in one short sentence: {q}") | student | StrOutputParser()

passes = 0
for row in golden:
    ans = answer_chain.invoke({"q": row["q"]}).strip()
    verdict = grade(row["q"], row["ref"], ans)
    passes += verdict == "PASS"
    print(f"[{verdict}] {row['q']}")
    print(f"        student said: {ans[:75]}")

print(f"\n📊 SCORE: {passes}/{len(golden)} passed")

[PASS] What is the capital of Australia?
        student said: The capital of Australia is Canberra.
[FAIL] How many letter 'e's are in the word 'excellence'?
        student said: The word "excellence" contains three 'e's.
[PASS] In what year did World War II end?
        student said: World War II ended in 1945.
[PASS] What is the chemical symbol for gold?
        student said: Au

📊 SCORE: 3/4 passed


## It Caught a Real Mistake

Look at the counting question — the student said **three** 'e's in "excellence"; it's actually **four** (remember Day 41: LLMs guess at counting). The judge flagged it **FAIL** on its own. That's the whole point: now you can measure quality across hundreds of questions without reading a single one by hand.

## One Honest Caveat

The judge is itself an LLM — so it can be wrong too. Real teams keep the reference answers tight, spot-check the judge, and track scores over time (LangSmith does this). An eval isn't perfect truth — it's a fast, repeatable signal that your AI got better or worse.

## Key Takeaways

1. **Evals = how you know your AI is good** — a golden dataset of questions + correct answers, scored automatically
2. **LLM-as-judge** — a stronger model grades another model's answers (PASS/FAIL), so you don't read thousands by hand
3. **The judge is an LLM too** — keep references tight, spot-check it, and track the score over time; it's a signal, not gospel

---

Next: Day 46 — Coming soon